In [1]:
# Import necessary libraries
import pandas as pd # For data manipulation and analysis
import numpy as np # For numerical operations
import pickle # For serializing and deserializing Python objects
from sklearn.metrics import roc_auc_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.append("/home/suraj/Repositories/TumorImagingBench/notebooks/modelling")

from modelling_utils import (
    train_knn_classifier, evaluate_model,
    train_linear_probing_classifier,
    train_few_shot_classifier,
    build_knn_ensemble_classifier, predict_with_ensemble,
    train_stacking_ensemble_classifier, predict_with_stacking_ensemble,
    plot_model_comparison, extract_model_features, 
    compute_knn_indices, compute_overlap_matrix, plot_overlap_matrix, 
    split_shuffle_data
)

In [2]:
# Load features from a pickle file
feature_dict_path = "/home/suraj/Repositories/TumorImagingBench/data/features/luna.pkl" # Path to the pickle file containing features
with open(feature_dict_path, 'rb') as file: # Open the file in read binary mode
    data = pickle.load(file) # Load the data from the pickle file

In [3]:
# Initialize shared variables
model_names = list(data.keys())
test_accuracies_dict = {}  # store baseline KNN AUCs with 95% CI


In [4]:
# Baseline KNN probing across models

for model_name, values in data.items():
    # Extract labels
    train_labels = [v["row"]["malignancy"] for v in values["train"]]
    val_labels = [v["row"]["malignancy"] for v in values["val"]]
    test_labels = [v["row"]["malignancy"] for v in values["test"]]

    # Stack features
    train_items = np.vstack([v["feature"] for v in values["train"]])
    val_items = np.vstack([v["feature"] for v in values["val"]])
    test_items = np.vstack([v["feature"] for v in values["test"]])

    # Combine splits for resampling
    all_items = np.vstack([train_items, val_items, test_items])
    all_labels = train_labels + val_labels + test_labels

    # Average across multiple shuffle splits
    n_splits = 10
    split_scores = []

    for split in range(n_splits):
        train_items_s, train_labels_s, val_items_s, val_labels_s, test_items_s, test_labels_s = split_shuffle_data(
            all_items, all_labels, train_ratio=0.5, val_ratio=0.2, random_seed=10 + split, stratify=True
        )

        best_model, study = train_knn_classifier(train_items_s, train_labels_s, val_items_s, val_labels_s)
        split_score = evaluate_model(best_model, test_items_s, test_labels_s)
        split_scores.append(split_score)

    avg_score = np.mean(split_scores)
    std_error = np.std(split_scores, ddof=1) / np.sqrt(n_splits)
    margin = 1.96 * std_error
    ci_lower = avg_score - margin
    ci_upper = avg_score + margin

    test_accuracies_dict[model_name] = {"mean": avg_score, "ci95": (ci_lower, ci_upper)}
    print(f"{model_name} AUC: {avg_score:.4f} ± {margin:.4f}")


[I 2025-11-25 14:15:38,685] A new study created in memory with name: no-name-68d5e131-ff41-477b-b902-e94361b7897b
[I 2025-11-25 14:15:38,695] Trial 0 finished with value: 0.5047619047619047 and parameters: {'k': 29}. Best is trial 0 with value: 0.5047619047619047.
[I 2025-11-25 14:15:38,710] Trial 1 finished with value: 0.49848484848484853 and parameters: {'k': 12}. Best is trial 0 with value: 0.5047619047619047.
[I 2025-11-25 14:15:38,719] Trial 2 finished with value: 0.5033549783549784 and parameters: {'k': 11}. Best is trial 0 with value: 0.5047619047619047.
[I 2025-11-25 14:15:38,752] Trial 3 finished with value: 0.5301948051948052 and parameters: {'k': 42}. Best is trial 3 with value: 0.5301948051948052.


[I 2025-11-25 14:15:38,773] Trial 4 finished with value: 0.5678571428571428 and parameters: {'k': 3}. Best is trial 4 with value: 0.5678571428571428.
[I 2025-11-25 14:15:38,799] Trial 5 finished with value: 0.5051948051948052 and parameters: {'k': 28}. Best is trial 4 with value: 0.5678571428571428.
[I 2025-11-25 14:15:38,809] Trial 6 finished with value: 0.5242424242424243 and parameters: {'k': 39}. Best is trial 4 with value: 0.5678571428571428.
[I 2025-11-25 14:15:38,820] Trial 7 finished with value: 0.49155844155844153 and parameters: {'k': 32}. Best is trial 4 with value: 0.5678571428571428.
[I 2025-11-25 14:15:38,850] Trial 8 finished with value: 0.5199134199134199 and parameters: {'k': 23}. Best is trial 4 with value: 0.5678571428571428.
[I 2025-11-25 14:15:38,892] Trial 9 finished with value: 0.5292207792207793 and parameters: {'k': 5}. Best is trial 4 with value: 0.5678571428571428.
[I 2025-11-25 14:15:38,969] Trial 10 finished with value: 0.4927489177489177 and parameters: {'

CTClipVitExtractor AUC: 0.5726 ± 0.0144


[I 2025-11-25 14:15:58,387] Trial 9 finished with value: 0.6726190476190477 and parameters: {'k': 5}. Best is trial 9 with value: 0.6726190476190477.
[I 2025-11-25 14:15:58,432] Trial 10 finished with value: 0.6192640692640693 and parameters: {'k': 34}. Best is trial 9 with value: 0.6726190476190477.
[I 2025-11-25 14:15:58,480] Trial 11 finished with value: 0.6108225108225109 and parameters: {'k': 36}. Best is trial 9 with value: 0.6726190476190477.
[I 2025-11-25 14:15:58,528] Trial 12 finished with value: 0.6216450216450217 and parameters: {'k': 27}. Best is trial 9 with value: 0.6726190476190477.
[I 2025-11-25 14:15:58,556] Trial 13 finished with value: 0.6133116883116883 and parameters: {'k': 35}. Best is trial 9 with value: 0.6726190476190477.
[I 2025-11-25 14:15:58,604] Trial 14 finished with value: 0.6433982683982684 and parameters: {'k': 19}. Best is trial 9 with value: 0.6726190476190477.
[I 2025-11-25 14:15:58,631] Trial 15 finished with value: 0.6305194805194806 and parameter

CTFMExtractor AUC: 0.6543 ± 0.0285


...

VISTA3DExtractor AUC: 0.7111 ± 0.0189


[I 2025-11-25 14:19:09,132] Trial 0 finished with value: 0.5279220779220779 and parameters: {'k': 29}. Best is trial 0 with value: 0.5279220779220779.
[I 2025-11-25 14:19:09,223] Trial 1 finished with value: 0.5374458874458874 and parameters: {'k': 12}. Best is trial 1 with value: 0.5374458874458874.
[I 2025-11-25 14:19:09,271] Trial 2 finished with value: 0.5566017316017315 and parameters: {'k': 11}. Best is trial 2 with value: 0.5566017316017315.
[I 2025-11-25 14:19:09,312] Trial 3 finished with value: 0.5403679653679654 and parameters: {'k': 42}. Best is trial 2 with value: 0.5566017316017315.
[I 2025-11-25 14:19:09,373] Trial 4 finished with value: 0.573051948051948 and parameters: {'k': 3}. Best is trial 4 with value: 0.573051948051948.
[I 2025-11-25 14:19:09,413] Trial 5 finished with value: 0.5088744588744588 and parameters: {'k': 28}. Best is trial 4 with value: 0.573051948051948.
[I 2025-11-25 14:19:09,445] Trial 6 finished with value: 0.5612554112554113 and parameters: {'k': 

VocoExtractor AUC: 0.5287 ± 0.0301


[I 2025-11-25 14:19:44,395] Trial 5 finished with value: 0.6148268398268398 and parameters: {'k': 28}. Best is trial 3 with value: 0.6773809523809523.
[I 2025-11-25 14:19:44,411] Trial 6 finished with value: 0.6821428571428572 and parameters: {'k': 39}. Best is trial 6 with value: 0.6821428571428572.
[I 2025-11-25 14:19:44,435] Trial 7 finished with value: 0.6602813852813854 and parameters: {'k': 32}. Best is trial 6 with value: 0.6821428571428572.
[I 2025-11-25 14:19:44,451] Trial 8 finished with value: 0.6082251082251083 and parameters: {'k': 23}. Best is trial 6 with value: 0.6821428571428572.
[I 2025-11-25 14:19:44,495] Trial 9 finished with value: 0.5891774891774891 and parameters: {'k': 5}. Best is trial 6 with value: 0.6821428571428572.
[I 2025-11-25 14:19:44,522] Trial 10 finished with value: 0.6772727272727275 and parameters: {'k': 34}. Best is trial 6 with value: 0.6821428571428572.
[I 2025-11-25 14:19:44,540] Trial 11 finished with value: 0.6703463203463204 and parameters: {

DummyResNetExtractor AUC: 0.6202 ± 0.0121


In [5]:
test_accuracies_dict

{'CTClipVitExtractor': {'mean': 0.5725637325637326,
  'ci95': (0.5581463368798577, 0.5869811282476075)},
 'CTFMExtractor': {'mean': 0.6543097643097643,
  'ci95': (0.6258466733983942, 0.6827728552211344)},
 'FMCIBExtractor': {'mean': 0.8860076960076959,
  'ci95': (0.8715736770355913, 0.9004417149798005)},
 'MerlinExtractor': {'mean': 0.6376430976430976,
  'ci95': (0.62534535058461, 0.6499408447015851)},
 'ModelsGenExtractor': {'mean': 0.806060606060606,
  'ci95': (0.795694502799469, 0.8164267093217431)},
 'PASTAExtractor': {'mean': 0.6647330447330447,
  'ci95': (0.6543163463915026, 0.6751497430745869)},
 'SUPREMExtractor': {'mean': 0.645021645021645,
  'ci95': (0.6190486808571002, 0.6709946091861899)},
 'VISTA3DExtractor': {'mean': 0.711120731120731,
  'ci95': (0.692220769541959, 0.7300206926995031)},
 'VocoExtractor': {'mean': 0.5287061087061088,
  'ci95': (0.49859526021378797, 0.5588169571984296)},
 'DummyResNetExtractor': {'mean': 0.6201731601731602,
  'ci95': (0.6080717242878912, 0.

In [6]:
# Plot test accuracies
fig = plot_model_comparison(test_accuracies_dict, font_size=30, height=1200, width=800, marker_color="#FCA308")
fig.show() # Show the plot
fig.write_image("luna_auc.png")

In [7]:
from pathlib import Path

if Path("overall_results.csv").exists():
    df = pd.read_csv("overall_results.csv")
else:
    df = pd.DataFrame()

df["Models"] = test_accuracies_dict.keys()
df["LUNA"] = [v["mean"] for k, v in test_accuracies_dict.items()]
df["LUNA_CI_Lower"] = [v["ci95"][0] for k, v in test_accuracies_dict.items()]
df["LUNA_CI_Upper"] = [v["ci95"][1] for k, v in test_accuracies_dict.items()]

df.to_csv("overall_results.csv")

In [8]:
model_features = extract_model_features(data)
model_neighbors = compute_knn_indices(model_features, num_neighbors=10, metric="cosine")
overlap_matrix, model_list = compute_overlap_matrix(model_neighbors)
fig = plot_overlap_matrix(overlap_matrix, model_list, font_size=30, tickangle=45)
fig.show()

fig.write_image("luna_knn.png")

## Linear Probing Evaluation

Evaluate foundation model features using linear probing (logistic regression).
This complements KNN probing and is the standard transfer learning baseline.


In [9]:
# Linear Probing - Train logistic regression on frozen features
linear_probing_results = {}

for model_name, values in data.items():
    print(f"Linear Probing - {model_name}...")
    
    # Extract labels
    train_labels = [v["row"]["malignancy"] for v in values["train"]]
    val_labels = [v["row"]["malignancy"] for v in values["val"]]
    test_labels = [v["row"]["malignancy"] for v in values["test"]]
    
    # Stack features
    train_items = np.vstack([v["feature"] for v in values["train"]])
    val_items = np.vstack([v["feature"] for v in values["val"]])
    test_items = np.vstack([v["feature"] for v in values["test"]])
    
    # Combine all splits
    all_items = np.vstack([train_items, val_items, test_items])
    all_labels = train_labels + val_labels + test_labels
    
    # Average across multiple shuffle splits
    n_splits = 10
    linear_split_scores = []
    
    for split in range(n_splits):
        # Get stratified split
        train_items_s, train_labels_s, val_items_s, val_labels_s, test_items_s, test_labels_s = split_shuffle_data(
            all_items, all_labels, train_ratio=0.5, val_ratio=0.2, random_seed=10+split, stratify=True
        )
        
        # Train linear probing classifier
        linear_model, _ = train_linear_probing_classifier(train_items_s, train_labels_s, val_items_s, val_labels_s)
        
        # Evaluate on test set
        linear_score = evaluate_model(linear_model, test_items_s, test_labels_s)
        linear_split_scores.append(linear_score)
    
    # Compute statistics
    avg_score = np.mean(linear_split_scores)
    std_error = np.std(linear_split_scores, ddof=1) / np.sqrt(n_splits)
    margin = 1.96 * std_error
    ci_lower = avg_score - margin
    ci_upper = avg_score + margin
    
    linear_probing_results[model_name] = {"mean": avg_score, "ci95": (ci_lower, ci_upper)}
    print(f"  Linear Probing AUC: {avg_score:.4f} ± {margin:.4f}")

print("\n✓ Linear probing evaluation complete")


Linear Probing - CTClipVitExtractor...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problem

  Linear Probing AUC: 0.5887 ± 0.0192
Linear Probing - CTFMExtractor...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problem

  Linear Probing AUC: 0.7502 ± 0.0191
Linear Probing - FMCIBExtractor...


...

  Linear Probing AUC: 0.8587 ± 0.0107
Linear Probing - VocoExtractor...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problem

  Linear Probing AUC: 0.5287 ± 0.0215
Linear Probing - DummyResNetExtractor...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problem

  Linear Probing AUC: 0.6377 ± 0.0142

✓ Linear probing evaluation complete


In [10]:
linear_probing_results

{'CTClipVitExtractor': {'mean': 0.5886676286676288,
  'ci95': (0.5695086107829017, 0.6078266465523559)},
 'CTFMExtractor': {'mean': 0.7502068302068302,
  'ci95': (0.7310608597376532, 0.7693528006760072)},
 'FMCIBExtractor': {'mean': 0.8941221741221742,
  'ci95': (0.8871674519300455, 0.9010768963143029)},
 'MerlinExtractor': {'mean': 0.6569600769600769,
  'ci95': (0.6411359794002209, 0.6727841745199328)},
 'ModelsGenExtractor': {'mean': 0.911053391053391,
  'ci95': (0.9046971839477835, 0.9174095981589985)},
 'PASTAExtractor': {'mean': 0.6854160654160654,
  'ci95': (0.663893632621453, 0.7069384982106778)},
 'SUPREMExtractor': {'mean': 0.6983838383838384,
  'ci95': (0.6810326869912073, 0.7157349897764695)},
 'VISTA3DExtractor': {'mean': 0.8586820586820586,
  'ci95': (0.8480185660111101, 0.8693455513530071)},
 'VocoExtractor': {'mean': 0.5286868686868688,
  'ci95': (0.50716284649982, 0.5502108908739175)},
 'DummyResNetExtractor': {'mean': 0.6377104377104377,
  'ci95': (0.6235131270561787, 

## Few-Shot Learning Evaluation

Evaluate foundation model generalization with limited training data (1-shot, 5-shot, 10-shot).
This assesses how well models work in clinical settings with limited labeled samples.


In [11]:
# Few-Shot Learning - Evaluate with limited training samples
shot_configs = [1, 5, 10]
few_shot_results = {shots: {} for shots in shot_configs}

for model_name, values in data.items():
    print(f"Few-Shot Learning - {model_name}...")
    
    # Extract labels
    train_labels = [v["row"]["malignancy"] for v in values["train"]]
    val_labels = [v["row"]["malignancy"] for v in values["val"]]
    test_labels = [v["row"]["malignancy"] for v in values["test"]]
    
    # Stack features
    train_items = np.vstack([v["feature"] for v in values["train"]])
    val_items = np.vstack([v["feature"] for v in values["val"]])
    test_items = np.vstack([v["feature"] for v in values["test"]])
    
    # Combine all splits
    all_items = np.vstack([train_items, val_items, test_items])
    all_labels = train_labels + val_labels + test_labels
    
    # Evaluate for different shot configurations
    for shots in shot_configs:
        n_splits = 10
        shot_scores = []
        
        for split in range(n_splits):
            # Get stratified split
            train_items_s, train_labels_s, val_items_s, val_labels_s, test_items_s, test_labels_s = split_shuffle_data(
                all_items, all_labels, train_ratio=0.5, val_ratio=0.2, random_seed=10+split, stratify=True
            )
            
            # Train few-shot model
            np.random.seed(split)
            few_shot_model, _, _ = train_few_shot_classifier(
                train_items_s, train_labels_s,
                val_items_s, val_labels_s,
                shots=shots
            )
            
            # Evaluate on test set
            test_score = evaluate_model(few_shot_model, test_items_s, test_labels_s)
            shot_scores.append(test_score)
        
        # Compute statistics
        mean_score = np.mean(shot_scores)
        std_error = np.std(shot_scores, ddof=1) / np.sqrt(n_splits)
        margin = 1.96 * std_error
        
        few_shot_results[shots][model_name] = {"mean": mean_score, "ci95": (mean_score - margin, mean_score + margin)}
        
        if shots == 1:  # Only print for 1-shot
            print(f"  {shots}-shot AUC: {mean_score:.4f} ± {margin:.4f} ... 10-shot: ", end="")
        elif shots == 10:
            print(f"{few_shot_results[shots][model_name]['mean']:.4f}")

print("\n✓ Few-shot learning evaluation complete")


[I 2025-11-25 14:24:18,430] A new study created in memory with name: no-name-4ffac4e5-0106-4756-9055-e3091f726907
[I 2025-11-25 14:24:18,436] Trial 0 finished with value: 0.5 and parameters: {'k': 2}. Best is trial 0 with value: 0.5.
[I 2025-11-25 14:24:18,439] Trial 1 finished with value: 0.48658008658008656 and parameters: {'k': 1}. Best is trial 0 with value: 0.5.
[I 2025-11-25 14:24:18,448] A new study created in memory with name: no-name-2f6562b7-4805-4d20-be06-6951030dc30b
[I 2025-11-25 14:24:18,452] Trial 0 finished with value: 0.5 and parameters: {'k': 2}. Best is trial 0 with value: 0.5.
[I 2025-11-25 14:24:18,456] Trial 1 finished with value: 0.48744588744588746 and parameters: {'k': 1}. Best is trial 0 with value: 0.5.
[I 2025-11-25 14:24:18,465] A new study created in memory with name: no-name-a6de1f8f-ae2a-4533-9a18-d09b6d648a23
[I 2025-11-25 14:24:18,469] Trial 0 finished with value: 0.5 and parameters: {'k': 2}. Best is trial 0 with value: 0.5.
[I 2025-11-25 14:24:18,473

Few-Shot Learning - CTClipVitExtractor...


[I 2025-11-25 14:24:18,549] A new study created in memory with name: no-name-89b84889-1bf0-4c6c-a287-7b2291d05412
[I 2025-11-25 14:24:18,554] Trial 0 finished with value: 0.5 and parameters: {'k': 2}. Best is trial 0 with value: 0.5.
[I 2025-11-25 14:24:18,560] Trial 1 finished with value: 0.5101731601731602 and parameters: {'k': 1}. Best is trial 1 with value: 0.5101731601731602.
[I 2025-11-25 14:24:18,572] A new study created in memory with name: no-name-0d8cb84c-bb39-45a4-ae49-e1965f3bd7ae
[I 2025-11-25 14:24:18,578] Trial 0 finished with value: 0.5 and parameters: {'k': 2}. Best is trial 0 with value: 0.5.
[I 2025-11-25 14:24:18,585] Trial 1 finished with value: 0.49523809523809526 and parameters: {'k': 1}. Best is trial 0 with value: 0.5.
[I 2025-11-25 14:24:18,597] A new study created in memory with name: no-name-9e3941e0-bee6-482a-9f84-67748ff59b46
[I 2025-11-25 14:24:18,607] Trial 0 finished with value: 0.5 and parameters: {'k': 2}. Best is trial 0 with value: 0.5.
[I 2025-11-2

  1-shot AUC: 0.4961 ± 0.0068 ... 10-shot: 

[I 2025-11-25 14:24:18,853] Trial 3 finished with value: 0.5 and parameters: {'k': 10}. Best is trial 0 with value: 0.5285714285714286.
[I 2025-11-25 14:24:18,858] Trial 4 finished with value: 0.47359307359307357 and parameters: {'k': 2}. Best is trial 0 with value: 0.5285714285714286.
[I 2025-11-25 14:24:18,873] Trial 5 finished with value: 0.49642857142857144 and parameters: {'k': 7}. Best is trial 0 with value: 0.5285714285714286.
[I 2025-11-25 14:24:18,894] Trial 6 finished with value: 0.47370129870129873 and parameters: {'k': 8}. Best is trial 0 with value: 0.5285714285714286.
[I 2025-11-25 14:24:18,907] Trial 7 finished with value: 0.4675324675324676 and parameters: {'k': 4}. Best is trial 0 with value: 0.5285714285714286.
[I 2025-11-25 14:24:18,918] Trial 8 finished with value: 0.4887445887445887 and parameters: {'k': 1}. Best is trial 0 with value: 0.5285714285714286.
[I 2025-11-25 14:24:18,929] Trial 9 finished with value: 0.4939393939393939 and parameters: {'k': 6}. Best is t

...

  1-shot AUC: 0.5036 ± 0.0134 ... 10-shot: 

[I 2025-11-25 14:25:03,585] Trial 5 finished with value: 0.5857142857142856 and parameters: {'k': 7}. Best is trial 4 with value: 0.5876623376623377.
[I 2025-11-25 14:25:03,597] Trial 6 finished with value: 0.5543290043290043 and parameters: {'k': 8}. Best is trial 4 with value: 0.5876623376623377.
[I 2025-11-25 14:25:03,602] Trial 7 finished with value: 0.5647186147186147 and parameters: {'k': 4}. Best is trial 4 with value: 0.5876623376623377.
[I 2025-11-25 14:25:03,617] Trial 8 finished with value: 0.5567099567099567 and parameters: {'k': 1}. Best is trial 4 with value: 0.5876623376623377.
[I 2025-11-25 14:25:03,622] Trial 9 finished with value: 0.5863636363636364 and parameters: {'k': 6}. Best is trial 4 with value: 0.5876623376623377.
[I 2025-11-25 14:25:03,669] A new study created in memory with name: no-name-5932ef28-39b3-4918-8af0-1fd7e3ff1699
[I 2025-11-25 14:25:03,678] Trial 0 finished with value: 0.5225108225108226 and parameters: {'k': 3}. Best is trial 0 with value: 0.5225

0.5102
Few-Shot Learning - DummyResNetExtractor...
  1-shot AUC: 0.4954 ± 0.0152 ... 10-shot: 

[I 2025-11-25 14:25:07,841] Trial 1 finished with value: 0.5497835497835497 and parameters: {'k': 9}. Best is trial 0 with value: 0.5737012987012987.
[I 2025-11-25 14:25:07,848] Trial 2 finished with value: 0.530952380952381 and parameters: {'k': 5}. Best is trial 0 with value: 0.5737012987012987.
[I 2025-11-25 14:25:07,862] Trial 3 finished with value: 0.5 and parameters: {'k': 10}. Best is trial 0 with value: 0.5737012987012987.
[I 2025-11-25 14:25:07,875] Trial 4 finished with value: 0.563961038961039 and parameters: {'k': 2}. Best is trial 0 with value: 0.5737012987012987.
[I 2025-11-25 14:25:07,885] Trial 5 finished with value: 0.4965367965367966 and parameters: {'k': 7}. Best is trial 0 with value: 0.5737012987012987.
[I 2025-11-25 14:25:07,901] Trial 6 finished with value: 0.5366883116883117 and parameters: {'k': 8}. Best is trial 0 with value: 0.5737012987012987.
[I 2025-11-25 14:25:07,912] Trial 7 finished with value: 0.5608225108225109 and parameters: {'k': 4}. Best is trial 

0.5256

✓ Few-shot learning evaluation complete


## Few-Shot Learning Curves

Visualize how model performance scales with increasing training samples.


In [12]:
# Plot few-shot learning curves

# The outer keys should be shot numbers, the inner keys should be model names.
model_names = list(next(iter(few_shot_results.values())).keys())
fig = go.Figure()

for model_name in model_names:
    shot_values = []
    means = []
    cis = []
    
    for shots in shot_configs:
        shot_values.append(shots)
        result = few_shot_results.get(shots, {}).get(model_name, {})
        if "mean" not in result or "ci95" not in result:
            means.append(float("nan"))
            cis.append(0)
        else:
            means.append(result["mean"])
            ci = result["ci95"][1] - result["mean"]
            cis.append(ci)
    
    fig.add_trace(go.Scatter(
        x=shot_values,
        y=means,
        error_y=dict(type='data', array=cis),
        mode='lines+markers',
        name=model_name,
        line=dict(width=2),
        marker=dict(size=8)
    ))

fig.update_layout(
    title='Few-Shot Learning Performance Scaling - LUNA Dataset',
    xaxis_title='Number of Shots per Class',
    yaxis_title='Test AUC',
    height=600,
    width=1000,
    template='simple_white',
    hovermode='x unified'
)
fig.update_xaxes(tickvals=shot_configs)
fig.update_yaxes(range=[0, 1.0])

fig.show()

print("Key Insight: Steeper curves = better low-data generalization")
print("Models with 1-shot performance close to 10-shot are most data-efficient")


Key Insight: Steeper curves = better low-data generalization
Models with 1-shot performance close to 10-shot are most data-efficient


In [13]:
few_shot_results[1]

{'CTClipVitExtractor': {'mean': 0.4961471861471861,
  'ci95': (0.4893353811398334, 0.5029589911545389)},
 'CTFMExtractor': {'mean': 0.5061038961038962,
  'ci95': (0.4872915721687957, 0.5249162200389966)},
 'FMCIBExtractor': {'mean': 0.6323809523809524,
  'ci95': (0.5577105338104544, 0.7070513709514503)},
 'MerlinExtractor': {'mean': 0.5258730158730158,
  'ci95': (0.5092389587904478, 0.5425070729555839)},
 'ModelsGenExtractor': {'mean': 0.5417460317460318,
  'ci95': (0.5098832487896194, 0.5736088147024441)},
 'PASTAExtractor': {'mean': 0.5045310245310246,
  'ci95': (0.4960365234745746, 0.5130255255874746)},
 'SUPREMExtractor': {'mean': 0.498051948051948,
  'ci95': (0.4881747779361438, 0.5079291181677523)},
 'VISTA3DExtractor': {'mean': 0.5225541125541125,
  'ci95': (0.49808387918855224, 0.5470243459196729)},
 'VocoExtractor': {'mean': 0.5035642135642135,
  'ci95': (0.4901639984222258, 0.5169644287062013)},
 'DummyResNetExtractor': {'mean': 0.4953535353535353,
  'ci95': (0.48012429519124

## Comparison: KNN vs Linear Probing vs Few-Shot


In [14]:
# Create comparison visualization
model_names = list(linear_probing_results.keys())

knn_means = [test_accuracies_dict[m]['mean'] for m in model_names]
linear_means = [linear_probing_results[m]['mean'] for m in model_names]
few_shot_10_means = [few_shot_results[10][m]['mean'] for m in model_names]

knn_errors = [test_accuracies_dict[m]['ci95'][1] - test_accuracies_dict[m]['mean'] for m in model_names]
linear_errors = [linear_probing_results[m]['ci95'][1] - linear_probing_results[m]['mean'] for m in model_names]
few_shot_errors = [few_shot_results[10][m]['ci95'][1] - few_shot_results[10][m]['mean'] for m in model_names]

# Create grouped bar chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=model_names,
    y=knn_means,
    error_y=dict(type='data', array=knn_errors),
    name='KNN Probing',
    marker_color='#4ECDC4'
))

fig.add_trace(go.Bar(
    x=model_names,
    y=linear_means,
    error_y=dict(type='data', array=linear_errors),
    name='Linear Probing',
    marker_color='#FF6B6B'
))

fig.add_trace(go.Bar(
    x=model_names,
    y=few_shot_10_means,
    error_y=dict(type='data', array=few_shot_errors),
    name='10-Shot Learning',
    marker_color='#95E1D3'
))

fig.update_layout(
    title='Evaluation Protocol Comparison - LUNA Dataset',
    xaxis_title='Model',
    yaxis_title='Test AUC',
    barmode='group',
    height=600,
    width=1400,
    template='simple_white',
    xaxis_tickangle=45
)
fig.update_yaxes(range=[0, 1.0])

fig.show()

# Print correlation analysis
print(f"Correlation KNN vs Linear Probing: {np.corrcoef(knn_means, linear_means)[0, 1]:.4f}")
print(f"Correlation KNN vs 10-Shot: {np.corrcoef(knn_means, few_shot_10_means)[0, 1]:.4f}")
print(f"\nInterpretation:")
print(f"  High KNN-Linear correlation (>0.8): Consistent feature quality assessment")
print(f"  10-Shot vs KNN: Difference shows data-efficiency gains with simple learning")


Correlation KNN vs Linear Probing: 0.9300
Correlation KNN vs 10-Shot: 0.9367

Interpretation:
  High KNN-Linear correlation (>0.8): Consistent feature quality assessment
  10-Shot vs KNN: Difference shows data-efficiency gains with simple learning


## Alignment-Based Ensemble Method

Combine all models using mutual k-NN overlap alignment as weights.
Models with high alignment with others are weighted more heavily.


In [15]:
# Build alignment-based ensemble
print("Building alignment-based ensemble...\n")

# Extract all features for ensemble
ensemble_features_dict = {}
for model_name, values in data.items():
    features_list = []
    for split in ['train', 'val', 'test']:
        if split in values and values[split]:
            split_features = np.vstack([v['feature'] for v in values[split]])
            features_list.append(split_features)
    ensemble_features_dict[model_name] = np.vstack(features_list) if features_list else np.array([])

# Compute alignment (already computed earlier, but re-using for clarity)
all_labels_ensemble = []
first_model = list(data.keys())[0]
for split in ['train', 'val', 'test']:
    split_labels = [v['row']['malignancy'] for v in data[first_model][split]]
    all_labels_ensemble.extend(split_labels)

# Evaluate ensemble across multiple splits
n_splits = 10
ensemble_scores = []
individual_ensemble_scores = {m: [] for m in model_list}

for split_idx in range(n_splits):
    # Split data
    (train_idx, train_labels_s, val_idx, val_labels_s, 
     test_idx, test_labels_s) = split_shuffle_data(
        np.arange(len(all_labels_ensemble)), all_labels_ensemble,
        train_ratio=0.5, val_ratio=0.2, random_seed=10+split_idx, stratify=True
    )
    
    # Get features for each split
    train_features_dict = {m: ensemble_features_dict[m][train_idx] for m in model_list}
    val_features_dict = {m: ensemble_features_dict[m][val_idx] for m in model_list}
    test_features_dict = {m: ensemble_features_dict[m][test_idx] for m in model_list}
    
    # Build ensemble
    ensemble_model, ensemble_val = build_knn_ensemble_classifier(
        train_features_dict, train_labels_s,
        val_features_dict, val_labels_s,
        overlap_matrix.copy(), model_list, k=10
    )
    
    # Evaluate ensemble on test
    ensemble_test_preds = predict_with_ensemble(ensemble_model, test_features_dict, model_list)
    
    if ensemble_test_preds.shape[1] == 2:
        ensemble_auc = roc_auc_score(test_labels_s, ensemble_test_preds[:, 1])
    else:
        ensemble_auc = roc_auc_score(test_labels_s, ensemble_test_preds, multi_class='ovr')
    
    ensemble_scores.append(ensemble_auc)
    
    # Evaluate individual models
    from sklearn.neighbors import KNeighborsClassifier
    for model_name in model_list:
        knn = KNeighborsClassifier(n_neighbors=10, metric='cosine')
        knn.fit(train_features_dict[model_name], train_labels_s)
        test_preds = knn.predict_proba(test_features_dict[model_name])
        
        if test_preds.shape[1] == 2:
            model_auc = roc_auc_score(test_labels_s, test_preds[:, 1])
        else:
            model_auc = roc_auc_score(test_labels_s, test_preds, multi_class='ovr')
        
        individual_ensemble_scores[model_name].append(model_auc)
    
    if (split_idx + 1) % 5 == 0:
        print(f"  Completed {split_idx + 1}/{n_splits} splits")

# Compute ensemble statistics
ensemble_mean = np.mean(ensemble_scores)
ensemble_std = np.std(ensemble_scores, ddof=1) / np.sqrt(n_splits)
ensemble_ci = 1.96 * ensemble_std

print(f"\n✓ Ensemble evaluation complete")
print(f"\nEnsemble Performance:")
print(f"  Test AUC: {ensemble_mean:.4f} ± {ensemble_ci:.4f}")

# Compare to individual models
print(f"\nComparison to Individual Models:")
best_model_name = None
best_model_score = 0
for model_name in model_list:
    ind_mean = np.mean(individual_ensemble_scores[model_name])
    ind_std = np.std(individual_ensemble_scores[model_name], ddof=1) / np.sqrt(n_splits)
    ind_ci = 1.96 * ind_std
    improvement = ensemble_mean - ind_mean
    
    if ind_mean > best_model_score:
        best_model_score = ind_mean
        best_model_name = model_name
    
    print(f"  {model_name}: {ind_mean:.4f} ± {ind_ci:.4f}  (ensemble: {improvement:+.4f})")

print(f"\nBest Single Model: {best_model_name} ({best_model_score:.4f})")
print(f"Ensemble Advantage: {ensemble_mean - best_model_score:+.4f}")


Building alignment-based ensemble...

  Completed 5/10 splits
  Completed 10/10 splits

✓ Ensemble evaluation complete

Ensemble Performance:
  Test AUC: 0.8188 ± 0.0097

Comparison to Individual Models:
  CTClipVitExtractor: 0.5670 ± 0.0206  (ensemble: +0.2518)
  CTFMExtractor: 0.6628 ± 0.0198  (ensemble: +0.1560)
  FMCIBExtractor: 0.8804 ± 0.0097  (ensemble: -0.0616)
  MerlinExtractor: 0.6388 ± 0.0139  (ensemble: +0.1799)
  ModelsGenExtractor: 0.7962 ± 0.0068  (ensemble: +0.0226)
  PASTAExtractor: 0.6653 ± 0.0135  (ensemble: +0.1535)
  SUPREMExtractor: 0.6053 ± 0.0308  (ensemble: +0.2135)
  VISTA3DExtractor: 0.7082 ± 0.0129  (ensemble: +0.1106)
  VocoExtractor: 0.5255 ± 0.0295  (ensemble: +0.2932)
  DummyResNetExtractor: 0.6034 ± 0.0126  (ensemble: +0.2154)

Best Single Model: FMCIBExtractor (0.8804)
Ensemble Advantage: -0.0616


## Ensemble vs Single Models

Bar plot comparing the alignment-weighted ensemble to each individual model.

In [ ]:
# Plot ensemble vs single-model performance
model_names_plot = list(individual_ensemble_scores.keys())
ind_means = [np.mean(individual_ensemble_scores[m]) for m in model_names_plot]
ind_errors = [1.96 * np.std(individual_ensemble_scores[m], ddof=1) / np.sqrt(len(individual_ensemble_scores[m])) for m in model_names_plot]

bar_names = model_names_plot + ["Ensemble"]
bar_means = ind_means + [ensemble_mean]
bar_errors = ind_errors + [ensemble_ci]

colors = ['#4ECDC4'] * len(model_names_plot) + ['#FCA308']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=bar_names,
    y=bar_means,
    error_y=dict(type='data', array=bar_errors),
    marker_color=colors,
    text=[f"{m:.3f}" for m in bar_means],
    textposition='auto'
))

fig.update_layout(
    title='Ensemble vs Individual Models',
    xaxis_title='Model',
    yaxis_title='Test AUC',
    height=600,
    width=1200,
    template='simple_white',
    xaxis_tickangle=45
)
fig.update_yaxes(range=[0, 1.0])

fig.show()

best_ind_mean = max(ind_means) if ind_means else float('nan')
print(f"Ensemble uplift over best single: {ensemble_mean - best_ind_mean:+.4f}")


## Ensemble Model Weights

Visualize the alignment-based weights assigned to each model.


In [16]:
# Display ensemble weights from the first evaluation
# (Run one more iteration to get final ensemble model)
train_idx, train_labels_s, val_idx, val_labels_s, test_idx, test_labels_s = split_shuffle_data(
    np.arange(len(all_labels_ensemble)), all_labels_ensemble,
    train_ratio=0.5, val_ratio=0.2, random_seed=50, stratify=True
)

train_features_dict = {m: ensemble_features_dict[m][train_idx] for m in model_list}
val_features_dict = {m: ensemble_features_dict[m][val_idx] for m in model_list}

final_ensemble, _ = build_knn_ensemble_classifier(
    train_features_dict, train_labels_s,
    val_features_dict, val_labels_s,
    overlap_matrix.copy(), model_list, k=10
)

# Extract and visualize weights
weights = final_ensemble['weights']
models_for_plot = list(weights.keys())
weight_values = list(weights.values())

fig = go.Figure()
fig.add_trace(go.Bar(
    x=models_for_plot,
    y=weight_values,
    marker_color='#FCA308',
    text=[f'{w:.3f}' for w in weight_values],
    textposition='auto',
))

fig.update_layout(
    title='Ensemble Model Weights (Based on k-NN Alignment)',
    xaxis_title='Model',
    yaxis_title='Weight',
    height=500,
    width=1200,
    template='simple_white',
    xaxis_tickangle=45
)
fig.update_yaxes(range=[0, max(weight_values) * 1.15])

fig.show()

print(f"\nWeight Statistics:")
print(f"  Max weight: {max(weight_values):.4f}")
print(f"  Min weight: {min(weight_values):.4f}")
print(f"  All weights sum to: {sum(weight_values):.4f}")
print(f"\nInterpretation:")
print(f"  Higher weight = better alignment with other models (consensus)")
print(f"  Lower weight = unique/diverse representation (adds complementary info)")



Weight Statistics:
  Max weight: 0.1570
  Min weight: 0.0232
  All weights sum to: 1.0000

Interpretation:
  Higher weight = better alignment with other models (consensus)
  Lower weight = unique/diverse representation (adds complementary info)


## Stacked Ensemble with Learned Weights

Train a logistic-regression meta-learner on top of per-model k-NN probabilities.

In [17]:
# Stacking ensemble with learned weights (meta-learned combination of models)
from sklearn.neighbors import KNeighborsClassifier

print("Evaluating stacking ensemble with learned weights...")

n_splits = 10
stacking_scores = []
stacking_val_scores = []
last_stacking_model = None
stacking_base_scores = {m: [] for m in model_list}

for split_idx in range(n_splits):
    (train_idx, train_labels_s, val_idx, val_labels_s,
     test_idx, test_labels_s) = split_shuffle_data(
        np.arange(len(all_labels_ensemble)), all_labels_ensemble,
        train_ratio=0.5, val_ratio=0.2, random_seed=110 + split_idx, stratify=True
    )

    train_features_dict = {m: ensemble_features_dict[m][train_idx] for m in model_list}
    val_features_dict = {m: ensemble_features_dict[m][val_idx] for m in model_list}
    test_features_dict = {m: ensemble_features_dict[m][test_idx] for m in model_list}

    stacking_model, val_auc = train_stacking_ensemble_classifier(
        train_features_dict, train_labels_s,
        val_features_dict, val_labels_s,
        k_candidates=(5, 10, 15, 25),
        meta_C_candidates=(0.25, 1.0, 4.0),
    )
    last_stacking_model = stacking_model
    stacking_val_scores.append(val_auc)

    test_pred = predict_with_stacking_ensemble(stacking_model, test_features_dict)
    if test_pred.shape[1] == 2:
        test_auc = roc_auc_score(test_labels_s, test_pred[:, 1])
    else:
        test_auc = roc_auc_score(test_labels_s, test_pred, multi_class='ovr')
    stacking_scores.append(test_auc)

    # Track single-model performance using the same k as the chosen base models
    k_for_base = stacking_model['k']
    for model_name in model_list:
        knn = KNeighborsClassifier(n_neighbors=k_for_base, metric="cosine")
        knn.fit(train_features_dict[model_name], train_labels_s)
        base_pred = knn.predict_proba(test_features_dict[model_name])
        if base_pred.shape[1] == 2:
            base_auc = roc_auc_score(test_labels_s, base_pred[:, 1])
        else:
            base_auc = roc_auc_score(test_labels_s, base_pred, multi_class='ovr')
        stacking_base_scores[model_name].append(base_auc)

    if (split_idx + 1) % 5 == 0:
        print(f"  Completed {split_idx + 1}/{n_splits} splits")

stacking_mean = np.mean(stacking_scores)
stacking_ci = 1.96 * np.std(stacking_scores, ddof=1) / np.sqrt(n_splits)
val_mean = np.mean(stacking_val_scores)

print(f"Stacking ensemble test AUC: {stacking_mean:.4f} ± {stacking_ci:.4f}")
print(f"Validation AUC (meta search average): {val_mean:.4f}")

best_single = None
best_single_score = -np.inf
for model_name, scores in stacking_base_scores.items():
    mean_score = np.mean(scores)
    if mean_score > best_single_score:
        best_single_score = mean_score
        best_single = model_name

print(f"Best single model (matched k): {best_single} — {best_single_score:.4f}")
print(f"Ensemble advantage over best single: {stacking_mean - best_single_score:+.4f}")


Evaluating stacking ensemble with learned weights...


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-package

  Completed 5/10 splits


/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning:

'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.

/home/suraj/Repositories/TumorImagingBench/.venv/lib/python3.11/site-package

  Completed 10/10 splits
Stacking ensemble test AUC: 0.9007 ± 0.0147
Validation AUC (meta search average): 0.9166
Best single model (matched k): FMCIBExtractor — 0.8954
Ensemble advantage over best single: +0.0053


In [19]:
# Visualize meta-learner weights from the last stacking run
if last_stacking_model is None:
    raise RuntimeError("Run the stacking ensemble cell before visualizing weights.")

meta_model = last_stacking_model['meta_model']
model_list = last_stacking_model['model_list']
coef = meta_model.coef_

n_models = len(model_list)
cols_per_model = coef.shape[1] // n_models

weight_rows = []
for idx, name in enumerate(model_list):
    start = idx * cols_per_model
    end = start + cols_per_model
    block = coef[0, start:end]
    weight_rows.append({
        "model": name,
        "meta_weight": float(np.mean(block))
    })

weight_df = pd.DataFrame(weight_rows)
weight_df['normalized'] = np.exp(weight_df['meta_weight']) / np.exp(weight_df['meta_weight']).sum()
weight_df = weight_df.sort_values('normalized', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=weight_df['model'],
    y=weight_df['normalized'],
    marker_color='#4ECDC4',
    text=[f"{w:.3f}" for w in weight_df['normalized']],
    textposition='auto'
))
fig.update_layout(
    title='Stacking Ensemble Meta-weights (softmax-normalized coefficients)',
    xaxis_title='Model',
    yaxis_title='Normalized weight',
    height=500,
    width=1200,
    template='simple_white',
    xaxis_tickangle=45,
)
fig.update_yaxes(range=[0, weight_df['normalized'].max() * 1.15])

fig.show()

print("Raw meta coefficients (per-model mean):")
print(weight_df[['model', 'meta_weight']].to_string(index=False))


Raw meta coefficients (per-model mean):
               model  meta_weight
      FMCIBExtractor     4.232421
  ModelsGenExtractor     3.192007
    VISTA3DExtractor     1.891480
       CTFMExtractor     0.349257
       VocoExtractor     0.049198
     SUPREMExtractor     0.030963
     MerlinExtractor    -0.004006
  CTClipVitExtractor    -1.305244
      PASTAExtractor    -1.576369
DummyResNetExtractor    -2.117991


## Ensemble Comparison Summary

Visualize alignment ensemble, stacked ensemble, and the best single model in one chart.

In [ ]:
# Compare ensembles against best single model
if 'ensemble_mean' not in globals() or 'stacking_mean' not in globals():
    raise RuntimeError("Run alignment and stacking sections first.")

best_single_name = best_model_name
best_single_mean = best_model_score
best_single_ci = 1.96 * np.std(individual_ensemble_scores[best_single_name], ddof=1) / np.sqrt(len(individual_ensemble_scores[best_single_name]))

labels = [f"Best single ({best_single_name})", "Alignment ensemble", "Stacking ensemble"]
means = [best_single_mean, ensemble_mean, stacking_mean]
errors = [best_single_ci, ensemble_ci, stacking_ci]
colors = ['#4ECDC4', '#FCA308', '#FF6B6B']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=labels,
    y=means,
    error_y=dict(type='data', array=errors),
    marker_color=colors,
    text=[f"{m:.3f}" for m in means],
    textposition='auto'
))

fig.update_layout(
    title='Alignment vs Stacking vs Best Single',
    xaxis_title='Model',
    yaxis_title='Test AUC',
    height=500,
    width=500,
    template='simple_white',
    xaxis_tickangle=20
)
fig.update_yaxes(range=[0, 1.0])
fig.show()

print(f"Stacking uplift over best single: {stacking_mean - best_single_mean:+.4f}")
print(f"Stacking uplift over alignment ensemble: {stacking_mean - ensemble_mean:+.4f}")


Stacking uplift over best single: +0.0203
Stacking uplift over alignment ensemble: +0.0819
